### Markdown 

Middleware provides a way to more tightly control what happens inside an agent . Middleware is useful for the folloing
- Tracking agent behavior with logging, analytics and debugging
- Transforming prompts , tool selection and output formatting
- Adding retries, fallbacks and termination logic 
- Applying rate limits, guard rails and PII Protection

In [37]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### Summerization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [38]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 5), # trigger when the length of LLM response reaches 5
            keep=("messages", 2) # use the last two messages to save as history

        )
    ]

)

In [39]:
### Run with thread id
# thread_id is a configurable identifier used by LangGraph checkpointers to associate graph state with a specific conversation or execution thread. 
# When a graph is invoked with the same thread_id, LangGraph retrieves the previously checkpointed state and continues from it, enabling conversational memory and long-running workflows.
config={"configurable":{"thread_id":"test-1"}}

In [40]:
# Alternative test data
questions=[
    " what is 2+2?",
    " what is 2%2?",
    " what is 2+10?",
    " what is 5+2?",
    " what is 12*2?",
    " what is 72+2?",
    " what is 92+27?",

]

from langchain_core.messages import HumanMessage

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")


Messages:{'messages': [HumanMessage(content=' what is 2+2?', additional_kwargs={}, response_metadata={}, id='cf3436a7-86e9-4676-9bb2-9e97848f6964'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_132d7ef4f6', 'id': 'chatcmpl-DoyPq1hBBKqVfr9ZvwI8ytgHzmzxK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eae51-1485-7a80-85ff-fb2c3ead232e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

In [41]:
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """Search hotels - retuens long response to use more tokens"""

    return f"""Hotels in city{c}
    1. Grand Hotel
    2. Novetel
    3.Holiday Inn
    4. Budget Stay"""

### Token Size

In [42]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent2 = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 2500), # trigger when the number of tokens>0500
            keep=("tokens", 100) # use the last two messages to save as history

        )
    ]

)

In [43]:
### Run with thread id
# thread_id is a configurable identifier used by LangGraph checkpointers to associate graph state with a specific conversation or execution thread. 
# When a graph is invoked with the same thread_id, LangGraph retrieves the previously checkpointed state and continues from it, enabling conversational memory and long-running workflows.
config={"configurable":{"thread_id":"test-2"}}

In [44]:
# Token count util

def count_token(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars / 4 # 4 chars = 1 token

In [45]:
cities=["Paris", "London", " Tokyo", "Dubai", "Hyderabad"]

from langchain_core.messages import HumanMessage

for c in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Find  hotels in the city{c}")]}, config)
    tokens = count_token(response["messages"])
    print(f"{c} : {tokens} tokens")
    print(f"{(response['messages'])}")

Paris : 355.0 tokens
[HumanMessage(content='Find  hotels in the cityParis', additional_kwargs={}, response_metadata={}, id='4158aa38-ae9b-4775-b77e-32ef2b0c533b'), AIMessage(content='Sure! Here are some popular hotels in Paris, France:\n\n1. **The Ritz Paris** - A luxury hotel located in Place Vendôme, known for its opulence and historic charm.\n\n2. **Hôtel de Crillon** - An elegant and historic hotel situated on the Place de la Concorde.\n\n3. **Le Meurice** - A luxurious hotel with beautiful views of the Tuileries Garden and a historic ambiance.\n\n4. **Shangri-La Hotel, Paris** - A former royal residence offering stunning views of the Eiffel Tower and luxurious amenities.\n\n5. **Park Hyatt Paris-Vendôme** - A modern and stylish hotel located near the Louvre and the Place Vendôme.\n\n6. **Hotel Le Palace** - A chic hotel located in the 10th arrondissement with a modern, artistic vibe.\n\n7. **Hôtel La Comtesse** - A boutique hotel close to the Eiffel Tower, offering beautiful views

​
### Human-in-the-loop
- Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [46]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [47]:
agent3 = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    tools=[read_email_tool ,send_email_tool],
    middleware=[
        
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]

)

In [48]:
config={"configurable":{"thread_id":"test_approve"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


{'messages': [HumanMessage(content="Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?", additional_kwargs={}, response_metadata={}, id='78f1ef06-107f-4592-a62e-295608e110d6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 99, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_800187fee2', 'id': 'chatcmpl-DoyQjymSB2q9mmQCtxU5efGvQlp4v', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eae51-eb82-7ec1-b738-85661fe28f3d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'johndoe@example.com', 'subject': 'Hello', 'body': 'How Are you?'}, 'id': 'call_

In [49]:
# Step 2: Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result=agent3.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"approve"
                    }
                ]
            }
        ),
        config=config
    )

    print (f"Result:{result['messages'][-1].content}")


 Paused Approving...
Result:The email has been successfully sent to johndoe@example.com with the subject "Hello" and the body "How Are you?".


### Reject

In [50]:
config={"configurable":{"thread_id":"test_reject"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


{'messages': [HumanMessage(content="Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?", additional_kwargs={}, response_metadata={}, id='4eadac77-1b8d-4303-9e01-84159ee044a5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 99, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_800187fee2', 'id': 'chatcmpl-DoyQkdlOHMauMdNDTDXLI0Bdn27Wl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eae51-f160-7e50-94af-8e71612847c5-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'johndoe@example.com', 'subject': 'Hello', 'body': 'How Are you?'}, 'id': 'call_

In [51]:
# Step 2: Reject

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result=agent3.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"reject"
                    }
                ]
            }
        ),
        config=config
    )

    print (f"Result:{result['messages'][-1].content}")

 Paused Approving...
Result:It seems there was an issue sending the email. Would you like to try again, or is there anything else I can assist you with?


### Edit

In [52]:
config={"configurable":{"thread_id":"test_edit"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


{'messages': [HumanMessage(content="Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?", additional_kwargs={}, response_metadata={}, id='09edec19-965d-4a7c-9827-2e2984391b10'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 99, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_800187fee2', 'id': 'chatcmpl-DoyQm5tjLRKTA6Gq1ylhrjeIUSZ4d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eae51-f72c-77b2-857f-9689a1085175-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'johndoe@example.com', 'subject': 'Hello', 'body': 'How Are you?'}, 'id': 'call_

In [53]:
# Step 2: Edit

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result = agent3.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "someone@example.com",
                            "subject": "Updated subject",
                            "body": "Updated email body"
                        }
                    }
                }
            ]
        }
    ),
    config )

    print (f"Result:{result['messages'][-1].content}")

 Paused Approving...
Result:


In [54]:
result

{'messages': [HumanMessage(content="Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?", additional_kwargs={}, response_metadata={}, id='09edec19-965d-4a7c-9827-2e2984391b10'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 99, 'total_tokens': 130, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_800187fee2', 'id': 'chatcmpl-DoyQm5tjLRKTA6Gq1ylhrjeIUSZ4d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eae51-f72c-77b2-857f-9689a1085175-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'someone@example.com', 'subject': 'Updated subject', 'body'